In [ ]:
!pip install -q langchain langchain-google-genai langchain-community faiss-cpu pypdf

import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('Ragproject')

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("Set- Up- Done")

Set- Up- Done


/tmp/ipykernel_1260/1875035598.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from google.colab import files
upload = files.upload()

pdf_name=list(upload.keys())[0]
print("uploaded pdf: ",pdf_name)

Saving employee_handbook.pdf to employee_handbook.pdf
uploaded pdf:  employee_handbook.pdf


In [ ]:
docs = PyPDFLoader(pdf_name).load()

print("Total number of pages in the document: ",len(docs))
print("Total number of characters in the pdf: ",sum(len(d.page_content)for d in docs))

Total number of pages in the document:  3
Total number of characters in the pdf:  5208


In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=150)

chunks = splitter.split_documents(docs)
print("Total number of chunks from the document: ",len(chunks))

Total number of chunks from the document:  12


In [ ]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks,embedding)

print("Totally",len(chunks),"chunks got stored inside the vectorstore")

Totally 12 chunks got stored inside the vectorstore


In [ ]:
retriever = vectorstore.as_retriever(searcg_kwargs={"k": 3})

found = retriever.invoke("How many days of sick leave are employees allowed per year?")

for d in found:
  print(d.page_content[:50])
  print("_"*50)

5. Sick Leave
 Employees may take up to 12 days o
__________________________________________________
 A work-from-home request should normally be subm
__________________________________________________
 Unused casual leave cannot be converted into cas
__________________________________________________
 Frequent late arrival without valid reasons may 
__________________________________________________


In [ ]:
for d, score in vectorstore.similarity_search_with_score("How many days of sick leave are employees allowed per year?",k=3):
  print(f"Distance: {score:.3f}  -> {d.page_content[:50]}")

Distance: 0.528  -> 5. Sick Leave
 Employees may take up to 12 days o
Distance: 0.601  ->  A work-from-home request should normally be subm
Distance: 0.672  ->  Unused casual leave cannot be converted into cas


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are an Employee Handbook Assistant. Answer the user's question using only the information provided in the context. "
         "If the answer cannot be found in the context, reply exactly: 'I could not find this in the employee handbook.' "
         "Do not use outside knowledge. Do not guess or make up information. "
         "If the question is unclear, answer only using information that can be directly supported by the context."),
        ("human",
         "Context from the document: {context}\n\nQuestion: {question}")
    ]
)

def formet_doc(docs):
  return "\n\n____\n\n".join(d.page_content for d in docs)

# Also fixing the spelling mistake from the retriever setup: 'searcg_kwargs' -> 'search_kwargs'
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {
        "context": retriever | formet_doc,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("chain ready")

chain ready


In [ ]:
print(rag_chain.invoke("How far in advance should a work-from-home request normally be submitted?"))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


A work-from-home request should normally be submitted at least one working day in advance.
